# Chapter 28: train() vs eval() & no_grad

        **Part V - The Training Loop, Mastered** · Companion notebook for *PyTorch From Ground Up, Volume I: Foundations*

        This notebook follows the chapter's progression with fresh, CPU-friendly examples. Type, change, break, and repair the code; the goal is fluency, not passive reading.

        ## Learning objectives

        - Switch layers between training and evaluation behavior
- Disable graph construction during validation
- Write a correct two-phase epoch


In [ ]:
import torch

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("Default device: cpu")


## Mode changes behavior

Dropout is random in train mode and inactive in eval mode. BatchNorm also changes how statistics are chosen.


In [ ]:
import torch.nn as nn

model = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Dropout(0.5), nn.Linear(16, 3))
x = torch.randn(4, 10)
model.train()
train_same = torch.allclose(model(x), model(x))
model.eval()
eval_same = torch.allclose(model(x), model(x))
print("train outputs same:", train_same, "eval outputs same:", eval_same)


## eval is not no_grad

eval changes module behavior; no_grad changes autograd behavior. Correct inference normally needs both.


In [ ]:
model.eval()
tracked = model(x)
with torch.no_grad():
    untracked = model(x)
print("eval only requires grad:", tracked.requires_grad)
print("eval + no_grad requires grad:", untracked.requires_grad)


## Reusable validation

A validation function sets eval mode, disables gradients, and computes a sample-weighted average.


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

labels = torch.randint(0, 3, (40,))
loader = DataLoader(TensorDataset(torch.randn(40, 10), labels), batch_size=12)
loss_fn = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    total = 0.0
    correct = 0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            logits = model(batch_x)
            total += loss_fn(logits, batch_y).item() * len(batch_x)
            correct += (logits.argmax(1) == batch_y).sum().item()
    return total / len(loader.dataset), correct / len(loader.dataset)

print("validation loss, accuracy:", evaluate(model, loader))


## Practice

        Work through these without looking back first.

        1. Show BatchNorm's train/eval difference.
2. Write a complete train-and-validation epoch.
3. Explain why no_grad alone does not disable Dropout.

        <details><summary>Study habit</summary>

        Predict every output shape before running a cell. When something fails, read the final line of the error and print the shape, dtype, and device of every tensor involved.

        </details>


In [ ]:
# Your practice space. Replace `pass` with your solution.
pass


## Recap

You completed Chapter 28's companion lab. Before moving on, explain the central idea aloud and recreate the smallest useful example from memory.

**Next:** return to the book for the full explanation, visual reasoning, watch-outs, and chapter exercises.
